In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/posts")

print(response.status_code)  # HTTP 상태 코드 출력
print(response.text)         # 응답 본문 출력

print('*'*50)
print(response.json())       # JSON 응답을 파싱하여 출력

df = pd.DataFrame(response.json())
df

# HTML 페이지 연결 : sample.html

Live-Server 웹서비스 구동 후 페이지 보기

In [ ]:
res = requests.get("http://127.0.0.1:5500/WebCrawl/sample.html")
res.text

soup = BeautifulSoup(res.text, 'html.parser')
print(soup.select('h1')[0].text)

print(soup.select('span'))

print('-'*50)
print(soup.select('div'))
print('-'*50)
soup.select('div#subject')  # id로 꺼내기  # => id tag
# soup.select_one('div').text
 

In [ ]:
soup.select('div.contents')  # class 명으로 꺼내기

HTML 속에 있는 요소를 검색하는 방법

1. 요소를 직접 지정 : find('요소명', id='id명')
2. CSS의 요소를 검색 : select('선택자')

CSS에서
- '#' 은 id, '.'은 class를 나타냄
- 자식은 '>'로 지정 : 'span>b'
- 후손은 공백으로 지정 ' span>a>b>c' => 'span c'
- 클래스속성에 css가 2개 정의되어 있고 두개를 모두 사용하고 싶을 때 '.'으로 나열   
&nbsp;&nbsp;&nbsp;&nbsp;예\) \<div class="head_info point_dn"\> => div.head_info.point_dn    
&nbsp;&nbsp;&nbsp;&nbsp;head_info와 point_dn을 '.'으로 연결   


In [ ]:
print(soup.select('div.contents>span>b')[0].text)
soup.select('div>span>b')[0].text
print(soup.select('div>b')[0].text)
print('-'*50)
print(soup.select('div:nth-of-type(2)')[0])

In [ ]:
marketindex = requests.get("https://finance.naver.com/marketindex/")
misoup = BeautifulSoup(marketindex.text, 'html.parser')

In [ ]:
#exchangeList > li.on > a.head.usd > div

usd = float(misoup.select('#exchangeList > li.on > a.head.usd > div > span.value')[0].text.replace(',', ''))
print("USD :", usd)

# JPY
# jpy = float(misoup.select('#worldExchangeList > li.on > a.head.jpy_usd > div > span.value')[0].text)
jpy = float(misoup.select('#worldExchangeList a.head.jpy_usd  span.value')[0].text)
print("JPY :", jpy)


#exchangeList > li:nth-child(2) > a.head.jpy > div > span.value
#worldExchangeList > li.on > a.head.jpy_usd > div > span.value

In [104]:
# YES24 Best Seller List
bestseller_columns = ['rank', 'title', 'author', 'price']    
bestseller_df = pd.DataFrame(columns=bestseller_columns)
    
    
for page in range(1,7) :  # Bestseller top 100
    url =  f"https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber={page}&pageSize=25"
    res = requests.get(url) 
    bestseller_soup = BeautifulSoup(res.text, 'html.parser')
    
    for seq in range(1,26) : 
        selector_rank = f"#yesBestList > li:nth-child({seq}) > div > div.item_img > div.img_canvas > div > em"
        selector_title = f'#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_name > a.gd_name'
        selector_price = f"#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_price > strong > em"
        selector_author = f"#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_pubGrp > span.authPub.info_auth"
        
        book_rank = int(bestseller_soup.select(selector_rank)[0].text)
        book_title = bestseller_soup.select(selector_title)[0].text
        book_author = bestseller_soup.select(selector_author)[0].text.strip()       
        book_price = int(bestseller_soup.select(selector_price)[0].text.replace(',',''))
        
        # bestseller = f"[{book_rank},{book_title},{book_author},{book_price}]"
        bestseller = [book_rank, book_title, book_author, book_price]
        bestseller_df.loc[book_rank] = bestseller
    
        # print(f"Rank {book_rank} : {book_title}, {book_author}, {book_price}원")

bestseller_df = bestseller_df.set_index('rank')

file_name = "yes24_best.csv"
bestseller_df.to_csv(file_name, encoding='utf-8-sig') # 한글깨짐 방지

bestseller_df.head()
bestseller_df.tail()

,title,author,price
rank,,,
146,눈물을 마시는 새 세트,이영도 저,59400
147,된다! 하루 만에 끝내는 제미나이 활용법,권서림 저,18000
148,데일 카네기 긍정태도론,데일 카네기 저/박선령 역,15210
149,명료함,탁민 오 저,17820
150,2026 문동균 한국사 한 권으로 모든 것을 정리하는 판서노트,문동균 편저,15300


In [2]:
class Book :
    def __init__(self, rank, title, author, price):
        self.rank = rank
        self.title = title
        self.author = author
        self.price = price
        
    def __init__(self, book):
        self.rank = book[0]
        self.title = book[1]
        self.author = book[2]
        self.price = book[3]
    
    def __str__(self) :
        return f"{self.rank:03d}: {self.title}, {self.author}, {self.price}원"


# YES24 Best Seller List
bestseller_columns = ['rank', 'title', 'author', 'price']    
bestseller_df = pd.DataFrame(columns=bestseller_columns)
    
    
for page in range(1,7) :  # Bestseller top 100
    url =  f"https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber={page}&pageSize=25"
    res = requests.get(url) 
    bestseller_soup = BeautifulSoup(res.text, 'html.parser')
    
    for seq in range(1,26) : 
        selector_rank = f"#yesBestList > li:nth-child({seq}) > div > div.item_img > div.img_canvas > div > em"
        selector_title = f'#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_name > a.gd_name'
        selector_price = f"#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_price > strong > em"
        selector_author = f"#yesBestList > li:nth-child({seq}) > div > div.item_info > div.info_row.info_pubGrp > span.authPub.info_auth"
        
        book_rank = int(bestseller_soup.select(selector_rank)[0].text)
        book_title = bestseller_soup.select(selector_title)[0].text
        book_author = bestseller_soup.select(selector_author)[0].text.strip()       
        book_price = int(bestseller_soup.select(selector_price)[0].text.replace(',',''))
        
        # bestseller = f"[{book_rank},{book_title},{book_author},{book_price}]"
        bestseller = [book_rank, book_title, book_author, book_price]
        bsbook = Book(bestseller)
        print(bsbook)
        bestseller_df.loc[book_rank] = bestseller
    
        # print(f"Rank {book_rank} : {book_title}, {book_author}, {book_price}원")

bestseller_df = bestseller_df.set_index('rank')

file_name = "yes24_best.csv"
bestseller_df.to_csv(file_name, encoding='utf-8-sig') # 한글깨짐 방지

bestseller_df.head()
bestseller_df.tail()

001: 나의 완벽한 장례식, 조현선 저, 17100원
002: 진보를 위한 주식투자, 이광수 저, 19800원
003: 박곰희 연금 부자 수업, 박곰희 저, 18900원
004: 괴테는 모든 것을 말했다, 스즈키 유이 저/이지수 역, 15300원
005: ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC, ETS 저, 19800원
006: ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC, ETS 저, 19800원
007: 죽은 왕녀를 위한 파반느 (양장 특별판), 박민규 저, 18000원
008: 자몽살구클럽, 한로로 저, 10800원
009: 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상, 최태성 저, 17100원
010: 너를 아끼며 살아라, 나태주 저, 16650원
011: 모순, 양귀자 저, 11700원
012: 돈의 방정식, 모건 하우절 저/박영준 역, 25200원
013: 자본주의 시대에서 살아남기 위한 최소한의 경제 공부, 백억남(김욱현) 저, 25200원
014: 1,000만 원으로 3년 안에 300만 원 월배당 만들기, 인생업(임승현) 저, 20700원
015: 위버멘쉬, 프리드리히 니체 저, 16020원
016: 엄마가 유령이 되었어!, 노부미 글그림/이기웅 역, 10800원
017: 안녕이라 그랬어, 김애란 저, 15120원
018: 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하, 최태성 저, 16650원
019: 최소한의 삼국지, 최태성 저/이성원 감수, 17550원
020: 싯다르타, 헤르만 헤세 저/박병덕 역, 7200원
021: 2026 심우철 실전 동형 모의고사 Season 2, 심우철 저, 13500원
022: 사이토 히토리의 어떻게 살 것인가, 사이토 히토리 저/황미숙 역, 11700원
023: 돈의 심리학 (50만 부 기념 뉴 에디션), 모건 하우절 저/이지연 역, 22320원
024: 더블 클릭, 알간지 저, 17100원
025: 사카모토 데이

,title,author,price
rank,,,
146,눈물을 마시는 새 세트,이영도 저,59400
147,된다! 하루 만에 끝내는 제미나이 활용법,권서림 저,18000
148,데일 카네기 긍정태도론,데일 카네기 저/박선령 역,15210
149,명료함,탁민 오 저,17820
150,2026 문동균 한국사 한 권으로 모든 것을 정리하는 판서노트,문동균 편저,15300


In [107]:
import sqlite3

conn = sqlite3.connect('_books.db')
cursor = conn.cursor()

# 테이블 생성
create_query = '''  
CREATE TABLE IF NOT EXISTS books (
    rank INTEGER PRIMARY KEY,
    title TEXT,
    author TEXT,
    price INTEGER
)
'''
cursor.execute(create_query)

conn.commit()
cursor.close()
conn.close()

In [105]:
for index, row in bestseller_df.iterrows():
    print(f"인덱스: {index}")
    print(f"이름: {row['title']}, 나이: {row['author']}")

인덱스: 1
이름: 나의 완벽한 장례식, 나이: 조현선 저
인덱스: 2
이름: 진보를 위한 주식투자, 나이: 이광수 저
인덱스: 3
이름: 박곰희 연금 부자 수업, 나이: 박곰희 저
인덱스: 4
이름: 괴테는 모든 것을 말했다, 나이: 스즈키 유이 저/이지수 역
인덱스: 5
이름: ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC, 나이: ETS 저
인덱스: 6
이름: ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC, 나이: ETS 저
인덱스: 7
이름: 죽은 왕녀를 위한 파반느 (양장 특별판), 나이: 박민규 저
인덱스: 8
이름: 자몽살구클럽, 나이: 한로로 저
인덱스: 9
이름: 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상, 나이: 최태성 저
인덱스: 10
이름: 너를 아끼며 살아라, 나이: 나태주 저
인덱스: 11
이름: 모순, 나이: 양귀자 저
인덱스: 12
이름: 돈의 방정식, 나이: 모건 하우절 저/박영준 역
인덱스: 13
이름: 자본주의 시대에서 살아남기 위한 최소한의 경제 공부, 나이: 백억남(김욱현) 저
인덱스: 14
이름: 1,000만 원으로 3년 안에 300만 원 월배당 만들기, 나이: 인생업(임승현) 저
인덱스: 15
이름: 위버멘쉬, 나이: 프리드리히 니체 저
인덱스: 16
이름: 엄마가 유령이 되었어!, 나이: 노부미 글그림/이기웅 역
인덱스: 17
이름: 안녕이라 그랬어, 나이: 김애란 저
인덱스: 18
이름: 2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하, 나이: 최태성 저
인덱스: 19
이름: 최소한의 삼국지, 나이: 최태성 저/이성원 감수
인덱스: 20
이름: 싯다르타, 나이: 헤르만 헤세 저/박병덕 역
인덱스: 21
이름: 2026 심우철 실전 동형 모의고사 Season 2, 나이: 심우철 저
인덱스: 22
이름: 사이토 히토리의 어떻게 살 것인가, 나이: 사이토 히토리 저/황미숙 역
인덱스: 23
이름: 돈의 심리학 (50만 부 기념 뉴 에디

In [ ]:
import sqlite3

conn = sqlite3.connect('_books.db')
cursor = conn.cursor()

# 데이터 삽입
insert_sql = "INSERT INTO books (rank, title, author, price) VALUES (?, ?, ?, ?)"

for index, book in bestseller_df.iterrows():
    cursor.execute(insert_sql, (index, book['title'], book['author'][:100], book['price']))

conn.commit()
cursor.close()
conn.close()

In [109]:
conn = sqlite3.connect('_books.db')
cursor = conn.cursor()

# 데이터 조회
cursor.execute('SELECT * FROM books')
rows = cursor.fetchall()
for row in rows:
    print(row)
    
cursor.close()
conn.close()

(1, '나의 완벽한 장례식', '조현선 저', 17100)
(2, '진보를 위한 주식투자', '이광수 저', 19800)
(3, '박곰희 연금 부자 수업', '박곰희 저', 18900)
(4, '괴테는 모든 것을 말했다', '스즈키 유이 저/이지수 역', 15300)
(5, 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC', 'ETS 저', 19800)
(6, 'ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC', 'ETS 저', 19800)
(7, '죽은 왕녀를 위한 파반느 (양장 특별판)', '박민규 저', 18000)
(8, '자몽살구클럽', '한로로 저', 10800)
(9, '2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상', '최태성 저', 17100)
(10, '너를 아끼며 살아라', '나태주 저', 16650)
(11, '모순', '양귀자 저', 11700)
(12, '돈의 방정식', '모건 하우절 저/박영준 역', 25200)
(13, '자본주의 시대에서 살아남기 위한 최소한의 경제 공부', '백억남(김욱현) 저', 25200)
(14, '1,000만 원으로 3년 안에 300만 원 월배당 만들기', '인생업(임승현) 저', 20700)
(15, '위버멘쉬', '프리드리히 니체 저', 16020)
(16, '엄마가 유령이 되었어!', '노부미 글그림/이기웅 역', 10800)
(17, '안녕이라 그랬어', '김애란 저', 15120)
(18, '2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하', '최태성 저', 16650)
(19, '최소한의 삼국지', '최태성 저/이성원 감수', 17550)
(20, '싯다르타', '헤르만 헤세 저/박병덕 역', 7200)
(21, '2026 심우철 실전 동형 모의고사 Season 2', '심우철 저', 13500)
(22, '사이토 히토리의 어떻게 살 것인가', '사이토 히토리 저/황미숙 역', 11700)
(23, '돈의 심리학

In [103]:
conn = sqlite3.connect('_books.db')
cursor = conn.cursor()

# 데이터 삭제
cursor.execute('DELETE FROM books')

conn.commit()    
cursor.close()
conn.close()

In [ ]:
import pymysql

try :
    con = pymysql.connect(
        host = 'localhost', # '127.0.0.1',
        user = 'root',
        password = '1234',
        database = 'wntrade',
        charset ='utf8'
    )
    cursor = con.cursor()
    print("접속 OK")
    print("작업 후 커서 및 MySQL 서버 접속 종료하세요")


except Exception as ex :
    print("접속 실패 :", ex)
    print("MySQL 서비스가 실행 중인지 확인 요망!!!")
    

# 테이블 생성
create_query = '''  
CREATE TABLE IF NOT EXISTS books (
    ranking INTEGER PRIMARY KEY,
    title VARCHAR(255),
    author VARCHAR(100),
    price INTEGER
)
'''
cursor.execute(create_query)

# 데이터 삽입
insert_sql = "INSERT INTO books (ranking, title, author, price) VALUES (%s, %s, %s, %s)"

for index, book in bestseller_df.iterrows():
    cursor.execute(insert_sql, (index, book['title'][:255], book['author'][:100], book['price']))

con.commit()

# 데이터 조회
cursor.execute('SELECT * FROM books')
rows = cursor.fetchall()
for row in rows:
    print(row)

cursor.close()
con.close()

접속 OK
작업 후 커서 및 MySQL 서버 접속 종료하세요
나의 완벽한 장례식
진보를 위한 주식투자
박곰희 연금 부자 수업
괴테는 모든 것을 말했다
ETS 토익 정기시험 기출문제집 1000 Vol. 5 RC
ETS 토익 정기시험 기출문제집 1000 Vol. 5 LC
죽은 왕녀를 위한 파반느 (양장 특별판)
자몽살구클럽
2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상
너를 아끼며 살아라
모순
돈의 방정식
자본주의 시대에서 살아남기 위한 최소한의 경제 공부
1,000만 원으로 3년 안에 300만 원 월배당 만들기
위버멘쉬
엄마가 유령이 되었어!
안녕이라 그랬어
2026 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하
최소한의 삼국지
싯다르타
2026 심우철 실전 동형 모의고사 Season 2
사이토 히토리의 어떻게 살 것인가
돈의 심리학 (50만 부 기념 뉴 에디션)
더블 클릭
사카모토 데이즈 25 더블특전판
부처님 말씀대로 살아보니
캔들차트 하나로 끝내는 추세추종 투자
코스피 1만 넥스트 레벨
혼모노
해커스 토익 기출 VOCA (보카)
나의 첫 월배당 ETF
이해찬 회고록
설민석의 한국사 대모험 36
코스모스
향기로운 꽃은 늠름하게 핀다 18 더블특전판
운명을 보는 기술
교사를 지키는 단단한 학급경영
어른의 행복은 조용하다
2026 써니 행정법총론 실전동형 모의고사
2026 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심화(1,2,3급)
주식투자 무작정 따라하기
절창 切創
아주 작은 습관의 힘 (50만 부 기념 스페셜 에디션)
2026 문동균 한국사 국가직 대비 실전 봉투 모의고사
2026 심우철 실전 동형 모의고사 Season 1
내가 돈을 벌고 있다는 착각
최소한의 한국사
내 심장을 쏴라
2026 선재국어 결승선 봉투 모의고사
2026 문동균 한국사 문단속 적중 최종병기 FINAL 모의고사
내 친구는 왜 그럴까?
급류
김재우의 기본 동사 100
다크 심리학
해커스 일본어 첫걸음 : 일본어 기초 